In [1]:
import numpy as np

class VectorizedHMMPOSTagger:
    """
    Hidden Markov Model (HMM) Part-of-Speech (POS) Tagger 
    implementing a fully vectorized Viterbi Algorithm.
    """
    def __init__(self, tags, vocabulary, init_prob, transition, emission):
        self.tags = tags
        self.vocabulary = vocabulary
        
        self.word_to_index = {word.lower(): i for i, word in enumerate(vocabulary)}
        
        # Log-space transformations to handle numerical underflow
        self.log_init = np.log(np.array(init_prob) + 1e-10)
        self.log_trans = np.log(np.array(transition) + 1e-10)
        self.log_emit = np.log(np.array(emission) + 1e-10)
        
    def viterbi(self, sentence):
        """
        Executes the fully vectorized Viterbi algorithm to find the most likely 
        sequence of hidden states (tags) for a given observation sequence (sentence).
        """
        T = len(sentence)
        N = len(self.tags)
        
        # Convert words to lowercase for vocabulary matching; fallback to 0 (first word) for unknowns
        sentence_idx = [self.word_to_index.get(word.lower(), 0) for word in sentence]
        
        # dp table to store the best log-probabilities
        dp = np.full((N, T), -np.inf)
        
        # backpointer table to trace back the best sequence of tags
        backpointer = np.zeros((N, T), dtype=int)
        
        # --- Initialization ---
        # Calculate probability for the first word
        dp[:, 0] = self.log_init + self.log_emit[:, sentence_idx[0]]
        
        # --- Fully Vectorized Recursion ---
        for t in range(1, T):
            # Vectorized computation of scores:
            # dp[:, t-1][:, np.newaxis] -> (N, 1) column vector of previous step scores
            # self.log_trans -> (N, N) transition matrix (rows: from_tag, cols: to_tag)
            # self.log_emit[:, sentence_idx[t]] -> (N,) array of emission scores for the current word
            
            # Broadcasting performs the nested loop additions entirely in C via NumPy
            scores = dp[:, t-1][:, np.newaxis] + self.log_trans + self.log_emit[:, sentence_idx[t]]
            
            # Maximize over previous tags (axis 0) for each current tag (axis 1)
            backpointer[:, t] = np.argmax(scores, axis=0)
            dp[:, t] = np.max(scores, axis=0)
            
        # --- Backtracking ---
        best_path = [np.argmax(dp[:, -1])]
        
        for t in range(T-1, 0, -1):
            best_path.append(backpointer[best_path[-1], t])
            
        best_path.reverse()
        
        return [self.tags[idx] for idx in best_path]

def evaluate_precision(predicted, ground_truth):
    correct = sum(1 for p, g in zip(predicted, ground_truth) if p == g)
    accuracy = correct / len(ground_truth)
    return accuracy

In [2]:
tags_1 = ["Noun", "Verb"]
words_1 = ["dogs", "bark"]
init_prob_1 = [0.6, 0.4]

transition_1 = [
    [0.7, 0.3], # Noun -> Noun, Verb
    [0.4, 0.6]  # Verb -> Noun, Verb
]

emission_1 = [
    [0.8, 0.2], # Noun -> dogs, bark
    [0.1, 0.9]  # Verb -> dogs, bark
]

tagger_1 = VectorizedHMMPOSTagger(tags_1, words_1, init_prob_1, transition_1, emission_1)
sentence_1 = ["Dogs", "Bark"]
predicted_1 = tagger_1.viterbi(sentence_1)

print(f"Sentence: {sentence_1}")
print(f"Predicted Tags: {predicted_1}")

Sentence: ['Dogs', 'Bark']
Predicted Tags: ['Noun', 'Verb']


In [3]:
tags_2 = ["Noun", "Verb", "Det", "Adj", "Prep"]
words_2 = ["the", "quick", "brown", "fox", "jumps", "over", "lazy", "dog"]

# Initial: Most likely to start with Determiner
init_prob_2 = [0.1, 0.05, 0.7, 0.1, 0.05]

# Transition Probabilities (Noun, Verb, Det, Adj, Prep)
transition_2 = [
    [0.1, 0.5, 0.05, 0.05, 0.3], # Noun  -> ...
    [0.2, 0.1, 0.3, 0.1, 0.3],   # Verb  -> ...
    [0.4, 0.05, 0.0, 0.55, 0.0], # Det   -> ...
    [0.7, 0.05, 0.05, 0.2, 0.0], # Adj   -> ...
    [0.1, 0.1, 0.6, 0.2, 0.0],   # Prep  -> ...
]

# Emission Probabilities (the, quick, brown, fox, jumps, over, lazy, dog)
emission_2 = [
    [0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.5], # Noun 
    [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0], # Verb 
    [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], # Det 
    [0.0, 0.4, 0.4, 0.0, 0.0, 0.0, 0.2, 0.0], # Adj 
    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0], # Prep 
]

tagger_2 = VectorizedHMMPOSTagger(tags_2, words_2, init_prob_2, transition_2, emission_2)

sentence_2 = ["The", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
ground_truth_2 = ["Det", "Adj", "Adj", "Noun", "Verb", "Prep", "Det", "Adj", "Noun"]

predicted_2 = tagger_2.viterbi(sentence_2)
precision = evaluate_precision(predicted_2, ground_truth_2)

print(f"Sentence: {sentence_2}")
print(f"Ground Truth:   {ground_truth_2}")
print(f"Predicted Tags: {predicted_2}")
print(f"Tagging Precision: {precision * 100:.2f}%")

Sentence: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
Ground Truth:   ['Det', 'Adj', 'Adj', 'Noun', 'Verb', 'Prep', 'Det', 'Adj', 'Noun']
Predicted Tags: ['Det', 'Adj', 'Adj', 'Noun', 'Verb', 'Prep', 'Det', 'Adj', 'Noun']
Tagging Precision: 100.00%
